# Eagle-I Impact Episode Exploration (temporary)

Exploratory-only notebook, separate from `geoe_eaglei_threshold_analysis.ipynb`. It does not
feed the multihazard database; it exists to look at the shape of the 2024-2025 Eagle-I impact
episode table before deciding what, if anything, becomes a permanent descriptor.

Covers:

- Histograms of episode size (customers, percent, integrated outage-hours) overall and by NERC
  Reliability Coordinator region.
- Power-law / heavy-tail inspection of episode size, duration, and empirical return period, plus
  pairwise cross-comparisons (duration vs. size, return period vs. size, return period vs.
  duration).
- A CONUS county-line heat map of per-county summary statistics, with NERC RC region boundaries
  overlaid.

**Requires `geopandas` and `shapely`** (only for the county-to-NERC-region spatial join in the
second code cell and the choropleth near the end). Run this notebook in your own Jupyter/VS Code
environment, not through an assistant sandbox -- these packages were not installable in the
sandbox used to write this notebook (no network/proxy access), so the geopandas-dependent cells
below are written carefully against the same conventions as
`geoe_eaglei_threshold_analysis.ipynb` cells 19-23, but were not executed or tested locally.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EAGLEI_DIR = Path('/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/outage_data/EAGLE-I')
EAGLEI_PROCESSED_DIR = EAGLEI_DIR / 'processed'
COUNTY_CACHE_DIR = EAGLEI_PROCESSED_DIR / 'county_geometry_cache'

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
NERC_GEOJSON_PATH = REPO_ROOT / 'data' / 'NERC_Reliability_Coordinators.geojson'

EAGLEI_EPISODES_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_impact_episodes_2024_2025.csv'
EAGLEI_EPISODE_OUTLIERS_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_impact_episode_outliers_2024_2025.csv'
COUNTY_INVENTORY_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_county_inventory.csv'
COUNTY_TO_NERC_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_county_to_nerc_region.csv'
COUNTY_TO_NERC_UNMAPPED_CSV = EAGLEI_PROCESSED_DIR / 'eaglei_county_to_nerc_region_unmapped.csv'

RUN_START = pd.Timestamp('2024-01-01')
RUN_END = pd.Timestamp('2025-12-31 23:59:59')
RECORD_LENGTH_YEARS = (RUN_END - RUN_START).total_seconds() / (365.25 * 24 * 3600)

print(f'Episodes source: {EAGLEI_EPISODES_CSV}  exists={EAGLEI_EPISODES_CSV.exists()}')
print(f'NERC geojson: {NERC_GEOJSON_PATH}  exists={NERC_GEOJSON_PATH.exists()}')
print(f'County geometry cache: {COUNTY_CACHE_DIR}  exists={COUNTY_CACHE_DIR.exists()}')
print(f'Record length: {RECORD_LENGTH_YEARS:.3f} years')


In [ ]:
# Load impact episodes and known outlier flags (informational only -- not excluded by default)

episodes = pd.read_csv(
    EAGLEI_EPISODES_CSV,
    parse_dates=['start_time', 'end_time', 'peak_time'],
    dtype={'county_fips': str},
)
episodes['county_fips'] = episodes['county_fips'].astype(str).str.zfill(5)
episodes['state_fips'] = episodes['county_fips'].str[:2]

if EAGLEI_EPISODE_OUTLIERS_CSV.exists():
    outliers = pd.read_csv(EAGLEI_EPISODE_OUTLIERS_CSV, dtype={'county_fips': str})
    flagged_ids = set(outliers['impact_episode_id'])
else:
    outliers = pd.DataFrame()
    flagged_ids = set()

episodes['is_flagged_outlier'] = episodes['impact_episode_id'].isin(flagged_ids)

print(f'Episodes loaded: {len(episodes):,}')
print(f'Counties represented: {episodes["county_fips"].nunique():,}')
print(f'Flagged as denominator-quirk outliers: {episodes["is_flagged_outlier"].sum():,} '
      f'({100 * episodes["is_flagged_outlier"].mean():.2f}%) -- kept in, not dropped, for this exploration')
episodes.head()


In [ ]:
# County-to-NERC-Reliability-Coordinator-region spatial join
#
# Follows the same conventions as geoe_eaglei_threshold_analysis.ipynb cells 19-23: county
# geometries from the cached Census cb_*_us_county_500k shapefiles (2024 primary, cascading to
# 2023/2022/2021 to recover counties missing from later vintages -- chiefly Connecticut, which
# switched from counties to planning regions around 2022 while Eagle-I still reports the legacy
# pre-planning-region county FIPS codes).
#
# Unlike the grid-cell mapping (many-to-many, a county can span several 50 km cells), each county
# should belong to exactly one NERC RC region, so this cell assigns each county to whichever NERC
# polygon it has the LARGEST area of overlap with (robust to counties that straddle an RC
# boundary), rather than a simple "within" predicate. Counties with no overlap at all (expected
# for Alaska, Hawaii, Puerto Rico, and the US Virgin Islands -- none of NERC's 13 RCs cover
# these) are reported as unmapped, not treated as an error.

try:
    import geopandas as gpd
except Exception as exc:
    raise ImportError('geopandas and shapely are required for the county-to-NERC-region join.') from exc


def _load_county_vintage(shp_name: str) -> gpd.GeoDataFrame:
    shp_path = COUNTY_CACHE_DIR / shp_name
    if not shp_path.exists():
        raise FileNotFoundError(
            f'{shp_path} not found. This notebook expects the county shapefile cache already '
            f'built by geoe_eaglei_threshold_analysis.ipynb cell 19 to be present.'
        )
    counties = gpd.read_file(shp_path)
    counties['county_fips'] = counties['GEOID'].astype(str).str.zfill(5)
    counties['state_fips'] = counties['STATEFP'].astype(str).str.zfill(2)
    counties['county_name'] = counties['NAME'].astype(str)
    return counties[['county_fips', 'state_fips', 'county_name', 'geometry']].to_crs('EPSG:4326')


def build_county_geometries_for_inventory(inventory_fips: set[str]) -> gpd.GeoDataFrame:
    pieces = []
    remaining = set(inventory_fips)
    for shp_name in (
        'cb_2024_us_county_500k.shp',
        'cb_2023_us_county_500k.shp',
        'cb_2022_us_county_500k.shp',
        'cb_2021_us_county_500k.shp',
    ):
        if not remaining:
            break
        vintage = _load_county_vintage(shp_name)
        matched = vintage[vintage['county_fips'].isin(remaining)].copy()
        if matched.empty:
            continue
        pieces.append(matched)
        remaining -= set(matched['county_fips'])
        print(f'{shp_name}: matched {len(matched)} counties ({len(remaining)} still unresolved)')
    if remaining:
        print(f'Counties with no geometry in any cached vintage: {len(remaining)} (e.g. {sorted(remaining)[:10]})')
    counties = pd.concat(pieces, ignore_index=True).drop_duplicates('county_fips') if pieces else \
        gpd.GeoDataFrame(columns=['county_fips', 'state_fips', 'county_name', 'geometry'])
    return gpd.GeoDataFrame(counties, geometry='geometry', crs='EPSG:4326')


def assign_counties_to_nerc_regions(
    counties: gpd.GeoDataFrame,
    nerc_regions: gpd.GeoDataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Equal-area CRS so overlap-area comparisons across the country are meaningful.
    counties_5070 = counties.to_crs('EPSG:5070')
    regions_5070 = nerc_regions.to_crs('EPSG:5070')[['NAME', 'ABBRV', 'geometry']].rename(
        columns={'NAME': 'nerc_region_name', 'ABBRV': 'nerc_region_abbrv'}
    )

    overlap = gpd.overlay(counties_5070, regions_5070, how='intersection')
    if overlap.empty:
        unmapped = counties[['county_fips', 'state_fips', 'county_name']].copy()
        unmapped['reason'] = 'no_overlap_with_any_nerc_rc_region'
        return pd.DataFrame(columns=['county_fips', 'state_fips', 'county_name', 'nerc_region_name', 'nerc_region_abbrv', 'overlap_area_km2', 'overlap_fraction_of_county']), unmapped

    overlap['overlap_area_km2'] = overlap.geometry.area / 1e6
    county_area_km2 = counties_5070.set_index('county_fips').geometry.area / 1e6

    best = (
        overlap.sort_values('overlap_area_km2', ascending=False)
        .drop_duplicates('county_fips', keep='first')
        .copy()
    )
    best['county_total_area_km2'] = best['county_fips'].map(county_area_km2)
    best['overlap_fraction_of_county'] = (best['overlap_area_km2'] / best['county_total_area_km2']).clip(upper=1.0)

    mapping = best[[
        'county_fips', 'state_fips', 'county_name',
        'nerc_region_name', 'nerc_region_abbrv',
        'overlap_area_km2', 'overlap_fraction_of_county',
    ]].sort_values('county_fips').reset_index(drop=True)

    mapped_fips = set(mapping['county_fips'])
    unmapped = counties.loc[~counties['county_fips'].isin(mapped_fips), ['county_fips', 'state_fips', 'county_name']].copy()
    unmapped['reason'] = 'no_overlap_with_any_nerc_rc_region'
    return mapping, unmapped


if COUNTY_TO_NERC_CSV.exists():
    county_to_nerc = pd.read_csv(COUNTY_TO_NERC_CSV, dtype={'county_fips': str, 'state_fips': str})
    unmapped_counties_nerc = pd.read_csv(COUNTY_TO_NERC_UNMAPPED_CSV, dtype={'county_fips': str, 'state_fips': str}) \
        if COUNTY_TO_NERC_UNMAPPED_CSV.exists() else pd.DataFrame()
    print(f'Loaded cached county-to-NERC-region mapping: {COUNTY_TO_NERC_CSV}')
else:
    county_inventory = pd.read_csv(COUNTY_INVENTORY_CSV, dtype={'county_fips': str, 'state_fips': str}) \
        if COUNTY_INVENTORY_CSV.exists() else episodes[['county_fips', 'state_fips', 'county_name', 'state_abbr']].drop_duplicates()
    nerc_regions = gpd.read_file(NERC_GEOJSON_PATH)

    counties_gdf = build_county_geometries_for_inventory(set(county_inventory['county_fips'].astype(str).str.zfill(5)))
    county_to_nerc, unmapped_counties_nerc = assign_counties_to_nerc_regions(counties_gdf, nerc_regions)

    county_to_nerc.to_csv(COUNTY_TO_NERC_CSV, index=False)
    unmapped_counties_nerc.to_csv(COUNTY_TO_NERC_UNMAPPED_CSV, index=False)
    print(f'Wrote county-to-NERC-region mapping: {COUNTY_TO_NERC_CSV}')

print(f'Counties mapped to a NERC RC region: {county_to_nerc["county_fips"].nunique():,}')
print(f'Counties unmapped (expected: AK, HI, PR, VI, and similar): {unmapped_counties_nerc["county_fips"].nunique() if not unmapped_counties_nerc.empty else 0:,}')
if not unmapped_counties_nerc.empty:
    display(unmapped_counties_nerc['state_fips'].value_counts())
county_to_nerc.head()


In [ ]:
# Attach NERC RC region to each episode

episodes = episodes.merge(
    county_to_nerc[['county_fips', 'nerc_region_name', 'nerc_region_abbrv']],
    on='county_fips',
    how='left',
)

n_unmapped = episodes['nerc_region_abbrv'].isna().sum()
print(f'Episodes without a NERC RC region (expected for AK/HI/PR/VI counties): {n_unmapped:,} '
      f'({100 * n_unmapped / len(episodes):.2f}%)')

nerc_region_order = (
    episodes.dropna(subset=['nerc_region_abbrv'])
    .groupby('nerc_region_abbrv')
    .size()
    .sort_values(ascending=False)
    .index.tolist()
)
print(f'NERC RC regions represented: {len(nerc_region_order)}')
episodes.groupby('nerc_region_abbrv').size().sort_values(ascending=False)


In [ ]:
# Histograms of episode size -- overall

SIZE_COLUMNS = {
    'peak_customers_out': 'Peak customers out',
    'peak_percent_out': 'Peak percent of county customers out',
    'duration_hours': 'Episode duration (hours)',
    'integrated_customer_outage_hours': 'Integrated customer-outage-hours',
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (col, label) in zip(axes.ravel(), SIZE_COLUMNS.items()):
    values = episodes[col].replace([np.inf, -np.inf], np.nan).dropna()
    values = values[values > 0]
    ax.hist(np.log10(values), bins=60, color='steelblue', edgecolor='none')
    ax.set_title(label)
    ax.set_xlabel(f'log10({label})')
    ax.set_ylabel('Episode count')
fig.suptitle(f'Eagle-I impact episode size distributions, 2024-2025 (n={len(episodes):,})')
fig.tight_layout()
plt.show()


In [ ]:
# Histograms of episode size by NERC RC region

plot_col = 'peak_customers_out'
n_regions = len(nerc_region_order)
n_cols = 4
n_rows = int(np.ceil(n_regions / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharex=True)
axes_flat = axes.ravel()
for ax, region in zip(axes_flat, nerc_region_order):
    subset = episodes.loc[episodes['nerc_region_abbrv'] == region, plot_col]
    subset = subset.replace([np.inf, -np.inf], np.nan).dropna()
    subset = subset[subset > 0]
    ax.hist(np.log10(subset), bins=30, color='darkorange', edgecolor='none')
    ax.set_title(f'{region} (n={len(subset):,})', fontsize=10)
for ax in axes_flat[n_regions:]:
    ax.axis('off')
fig.suptitle(f'log10({plot_col}) by NERC Reliability Coordinator region')
fig.tight_layout()
plt.show()


In [ ]:
# Power-law / heavy-tail inspection: empirical CCDF (log-log) with a tail-slope fit
#
# Simple rank-based CCDF, not a formal MLE power-law fit (no `powerlaw` package assumed
# available) -- P(X >= x) estimated as rank/N on values sorted descending, tail slope estimated
# by ordinary least squares on log10(x) vs. log10(CCDF) over the top `tail_fraction` of events.

def empirical_ccdf(values: pd.Series) -> pd.DataFrame:
    values = values.replace([np.inf, -np.inf], np.nan).dropna()
    values = values[values > 0].sort_values(ascending=False).reset_index(drop=True)
    n = len(values)
    ccdf = (np.arange(1, n + 1)) / n
    return pd.DataFrame({'value': values.values, 'ccdf': ccdf})


def fit_tail_slope(ccdf_table: pd.DataFrame, tail_fraction: float = 0.1) -> float:
    n_tail = max(int(len(ccdf_table) * tail_fraction), 10)
    tail = ccdf_table.iloc[:n_tail]
    log_x = np.log10(tail['value'])
    log_y = np.log10(tail['ccdf'])
    slope, intercept = np.polyfit(log_x, log_y, 1)
    return slope


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
tail_slopes = {}
for ax, col in zip(axes, ['peak_customers_out', 'duration_hours', 'integrated_customer_outage_hours']):
    table = empirical_ccdf(episodes[col])
    slope = fit_tail_slope(table)
    tail_slopes[col] = slope
    ax.loglog(table['value'], table['ccdf'], marker='.', linestyle='none', markersize=2, alpha=0.4)
    ax.set_xlabel(col)
    ax.set_ylabel('P(X >= x)')
    ax.set_title(f'{col}\ntail slope (top 10%): {slope:.2f}')
fig.suptitle('Empirical CCDFs, log-log (steeper negative slope = lighter tail)')
fig.tight_layout()
plt.show()

print('Fitted tail slopes (top 10% of events):')
for col, slope in tail_slopes.items():
    print(f'  {col}: {slope:.3f}')


In [ ]:
# Empirical return periods (Weibull plotting position), pooled nationally and by NERC region
#
# return_period_years = record_length_years / exceedance_probability, using the Weibull plotting
# position rank/(N+1) for the exceedance probability, applied to each size metric independently
# (this treats every episode row -- one per county per event -- as a distinct occurrence; it is
# NOT a per-county return period, it's a "how often does an episode of this size occur somewhere
# in the record" pooled statistic).
#
# 2026-09-22 fix: an earlier version of this cell built a separate rank table per column and
# merged it back onto `episodes` by matching on the raw value (e.g. `on='peak_customers_out'`).
# That crashed the kernel: many episodes share identical duration_hours/peak_customers_out values
# (zeros and small integers are common), so merging on value equality is a many-to-many join that
# can multiply 38k rows into tens of millions. Fixed by computing the rank in place with
# `Series.rank()`, which needs no merge at all.

def add_return_period_column(df: pd.DataFrame, value_col: str, new_col: str, record_length_years: float) -> pd.DataFrame:
    valid = df[value_col].replace([np.inf, -np.inf], np.nan)
    valid = valid.where(valid > 0)
    n = int(valid.notna().sum())
    ranks = valid.rank(method='average', ascending=False)
    exceedance_prob = ranks / (n + 1)
    df[new_col] = record_length_years / exceedance_prob
    return df


RETURN_PERIOD_COLUMNS = ['peak_customers_out', 'duration_hours', 'integrated_customer_outage_hours']
for col in RETURN_PERIOD_COLUMNS:
    episodes = add_return_period_column(episodes, col, f'return_period_years_{col}', RECORD_LENGTH_YEARS)

fig, axes = plt.subplots(1, len(RETURN_PERIOD_COLUMNS), figsize=(6 * len(RETURN_PERIOD_COLUMNS), 5))
for ax, col in zip(axes, RETURN_PERIOD_COLUMNS):
    rp_col = f'return_period_years_{col}'
    plot_df = episodes[[rp_col, col]].dropna()
    ax.loglog(plot_df[rp_col], plot_df[col], marker='.', linestyle='none', markersize=2, alpha=0.4)
    ax.set_xlabel('Empirical return period (years, pooled nationally)')
    ax.set_ylabel(col)
    ax.set_title(col)
fig.suptitle(f'Event size vs. empirical return period (record length {RECORD_LENGTH_YEARS:.2f} yrs)')
fig.tight_layout()
plt.show()


In [ ]:
# Pairwise cross-comparisons: duration vs. size, return period vs. size, return period vs. duration
#
# 2026-09-22 fix: this cell previously built its own `merged` DataFrame via two more value-keyed
# merges on top of the already-crash-prone return-period merges above -- same many-to-many blowup
# risk, compounded twice. The return-period columns now already live directly on `episodes` (see
# the previous cell), so no merge is needed here at all -- just select and drop NaNs/zeros.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
plot_df = episodes[['duration_hours', 'peak_customers_out']].replace(0, np.nan).dropna()
ax.loglog(plot_df['duration_hours'], plot_df['peak_customers_out'], marker='.', linestyle='none', markersize=2, alpha=0.3)
ax.set_xlabel('Duration (hours)')
ax.set_ylabel('Peak customers out')
ax.set_title('Duration vs. size')

ax = axes[1]
plot_df = episodes[['return_period_years_peak_customers_out', 'peak_customers_out']].replace(0, np.nan).dropna()
ax.loglog(plot_df['return_period_years_peak_customers_out'], plot_df['peak_customers_out'],
          marker='.', linestyle='none', markersize=2, alpha=0.3, color='darkgreen')
ax.set_xlabel('Return period (years, by size)')
ax.set_ylabel('Peak customers out')
ax.set_title('Return period vs. size')

ax = axes[2]
plot_df = episodes[['return_period_years_duration_hours', 'duration_hours']].replace(0, np.nan).dropna()
ax.loglog(plot_df['return_period_years_duration_hours'], plot_df['duration_hours'],
          marker='.', linestyle='none', markersize=2, alpha=0.3, color='darkred')
ax.set_xlabel('Return period (years, by duration)')
ax.set_ylabel('Duration (hours)')
ax.set_title('Return period vs. duration')

fig.tight_layout()
plt.show()


In [ ]:
# CONUS county-line heat map of per-county summary statistics, with NERC RC boundaries overlaid
#
# Requires the same geopandas/shapely stack as the county-to-NERC-region join above. Reuses the
# same cached county geometries built there; excludes Alaska/Hawaii/territories from this
# specific map since a CONUS-only choropleth is the most legible for a first look.

county_summary = (
    episodes.groupby('county_fips')
    .agg(
        n_episodes=('impact_episode_id', 'size'),
        total_customer_outage_hours=('integrated_customer_outage_hours', 'sum'),
        max_peak_percent_out=('peak_percent_out', 'max'),
        max_peak_customers_out=('peak_customers_out', 'max'),
    )
    .reset_index()
)

counties_conus = build_county_geometries_for_inventory(set(county_summary['county_fips']))
counties_conus = counties_conus[~counties_conus['state_fips'].isin(['02', '15', '72', '78', '60', '66', '69'])].copy()
counties_conus = counties_conus.merge(county_summary, on='county_fips', how='left')

nerc_regions_conus = gpd.read_file(NERC_GEOJSON_PATH)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

counties_conus.plot(
    column='n_episodes', cmap='viridis', linewidth=0.1, edgecolor='grey',
    legend=True, ax=axes[0], missing_kwds={'color': 'lightgrey'},
)
nerc_regions_conus.boundary.plot(ax=axes[0], color='black', linewidth=0.8)
axes[0].set_title('Impact episodes per county, 2024-2025')
axes[0].axis('off')

counties_conus.plot(
    column='total_customer_outage_hours', cmap='magma', linewidth=0.1, edgecolor='grey',
    legend=True, ax=axes[1], missing_kwds={'color': 'lightgrey'},
    scheme=None,
)
nerc_regions_conus.boundary.plot(ax=axes[1], color='white', linewidth=0.8)
axes[1].set_title('Total integrated customer-outage-hours per county, 2024-2025')
axes[1].axis('off')

fig.tight_layout()
plt.show()


## Notes and limitations

- This notebook is exploratory scaffolding, not a validated science product -- nothing here has
  been sanity-checked the way `eaglei_full_period_regen_checklist.md` requires for the main
  threshold-analysis notebook's outputs.
- Alaska, Hawaii, Puerto Rico, and the US Virgin Islands are not covered by any of NERC's 13
  Reliability Coordinator regions, so their counties are expected to show up in
  `eaglei_county_to_nerc_region_unmapped.csv` -- this is normal, not a join bug.
- The "return period" here is a pooled national empirical statistic (Weibull plotting position
  over all county-episodes together), not a per-county return period. A per-county version would
  need each county's own record treated separately, which will have far fewer events per county
  and much noisier tail estimates given only a 2-year record.
- The tail-slope fit in the power-law cell is an OLS fit on log-log data over the top decile, not
  a maximum-likelihood power-law fit (e.g. via the `powerlaw` package) -- treat the slopes as a
  rough heavy-tail indicator, not a rigorous power-law exponent estimate.
- `flag_percent_out_gt_100` / `flag_customers_out_gt_county_total` outlier episodes (see
  `eaglei_impact_episode_outliers_2024_2025.csv`) are flagged (`is_flagged_outlier`) but were
  deliberately NOT excluded from any plot above -- filter them out first if they turn out to
  distort a specific figure.
